# Protein Generation Using Large Reasoning Models for Silkome Reasoning

By: Jay Siri (jaysiri@mit.edu)

This notebook uses finetuned LLMs that reason about silk fiber mechanical properties to generate proteins, then characterizes them.

**Characterization:**
- BLAST for novelty assessment
- ProtParam for primary structure characterization
  - Compare properties to distribution of Silkome train
- (AlphaFold3 for PDB structure prediction) &rarr; DSSP for secondary structure characterization


### **Set Up and Load Model**

In [ ]:
!pip install Bio
!pip install -q biopython
!pip install torchao --upgrade
!apt install ncbi-blast+
!apt-get install dssp

In [ ]:
import os, re, json, random
import torch
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import glob

from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from datasets import load_dataset
from peft import PeftModel
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.PDB import PDBParser, DSSP
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from typing import Optional, Tuple, Dict, List, Any
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
SEED = int(os.environ.get("SEED", "42"))
set_seed(SEED)
random.seed(SEED)

In [ ]:
# Choose a Hugging Face pretrained model name
ADAPTER_NAME ='jayyysiri/silkome_reasoning_v1.0' # Public model adapter name from HF
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)
model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, ADAPTER_NAME)

### **Generate Proteins**

In [ ]:
GENERATIVE_PROTEIN_PROMPT = """Generate a spidroin protein that has high strength and toughness properties. Return the only sequence in FASTA format."""

def generate_protein():
  generated = torch.tensor(tokenizer.encode(GENERATIVE_PROTEIN_PROMPT)).unsqueeze(0).to(model.device)
  sample_outputs = model.generate(
                    inputs=generated,
                    max_new_tokens=1500,
                    min_new_tokens=40,
                    pad_token_id=tokenizer.pad_token_id,
                    temperature=0.75,
                    use_cache=False
                    )
  decoded_text = tokenizer.decode(sample_outputs[-1], skip_special_tokens=True)
  return decoded_text

In [ ]:
N_TO_GENERATE = 5

generated_sequences = []
for i in range(N_TO_GENERATE):
  generated_sequences.append(generate_protein())

### **BLAST**

Do BLAST searches against the nr database to check novelty.

In [ ]:
# -----------------------------
# Novelty assessment with BLAST
# -----------------------------
for i, seq in enumerate(generated_sequences):
  print(f"Processing sequence {i} ...")
  result_handle = NCBIWWW.qblast("blastp", "nr", seq)

  with open(f"blast_results_{i}.xml", "w") as out_handle:
      out_handle.write(result_handle.read())

  result_handle.close()

In [ ]:
def parse_blast_file(file):
    """Helper to read NCBIXML file and do sequence searching"""
    print(f"Parsing {file} ...")
    results = []
    with open(file) as handle:
        blast_records = NCBIXML.parse(handle)
        for record in blast_records:
            if not record.alignments:
                results.append((file, "NOVEL", None, None, None))
            else:
                # take the best hit
                top_alignment = record.alignments[0]
                hsp = top_alignment.hsps[0]
                results.append((
                    file,
                    "HIT",
                    f"Match: {top_alignment.title}",
                    f"E-value: {hsp.expect}" ,
                    f"Identity: {hsp.identities / hsp.align_length}"
                ))
    return results


# Concurrently use NCBIWWW server to run BLAST
files = glob.glob("blast_results_*.xml")
all_results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(parse_blast_file, f) for f in files]
    for future in as_completed(futures):
        all_results.extend(future.result())

# Print results
for r in all_results:
    print(r)

### **ProtParam**
Characterize and visualize properties of the primary structure using ProtParam: MW, aromaticity, instability index, isoelectric point, GRAVY metric, etc.

In [ ]:
# Load Silkome dataset to compare against
SILKOME_TRAIN_URL = "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/train/data-00000-of-00001.arrow"
SILKOME_TEST_URL= "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/test/data-00000-of-00001.arrow"
silk = load_dataset("arrow", data_files={'train': SILKOME_TRAIN_URL, 'test': SILKOME_TEST_URL})
silk_train = silk.get("train")

def silkome_seqs_extract(ex):
    input = ex["input"]
    output = eval(ex["output"])
    seq = re.search(r"Protein sequence: \s*([A-Za-z]+)", input)
    seq = seq.group(0)[len("Protein sequence: "):]
    return seq

training_sequences = []
for item in silk_train:
    training_sequences.append(silkome_seqs_extract(item))

In [ ]:
# -------------------------------------------------
# Primary Structure Characterization with ProtParam
# -------------------------------------------------
training_seq_features = {
    "Molecular Weight": [],
    "Aromaticity": [],
    "Instability": [],
    "Mean Flexibility": [],
    "pI": [],
    "GRAVY": [],
    "Helix Fraction": [],
    "Sheet Fraction": [],
    "Coil Fraction": []
}

generated_seq_features = {
    "Molecular Weight": [],
    "Aromaticity": [],
    "Instability": [],
    "Mean Flexibility": [],
    "pI": [],
    "GRAVY": [],
    "Helix Fraction": [],
    "Sheet Fraction": [],
    "Coil Fraction": []
}

for seq in training_sequences:
    try:
      analysis = ProteinAnalysis(seq)
      training_seq_features["Molecular Weight"].append(analysis.molecular_weight())
      training_seq_features["Aromaticity"].append(analysis.aromaticity())
      training_seq_features["Instability"].append(analysis.instability_index())
      training_seq_features["Mean Flexibility"].append(np.mean(analysis.flexibility()))
      training_seq_features["pI"].append(analysis.isoelectric_point())
      training_seq_features["GRAVY"].append(analysis.gravy())

      helix, sheet, coil = analysis.secondary_structure_fraction()
      training_seq_features["Helix Fraction"].append(helix)
      training_seq_features["Sheet Fraction"].append(sheet)
      training_seq_features["Coil Fraction"].append(coil)
    except:
      pass

for seq in generated_sequences:
    try:
      analysis = ProteinAnalysis(seq)
      generated_seq_features["Molecular Weight"].append(analysis.molecular_weight())
      generated_seq_features["Aromaticity"].append(analysis.aromaticity())
      generated_seq_features["Instability"].append(analysis.instability_index())
      generated_seq_features["Mean Flexibility"].append(np.mean(analysis.flexibility()))
      generated_seq_features["pI"].append(analysis.isoelectric_point())
      generated_seq_features["GRAVY"].append(analysis.gravy())

      helix, sheet, coil = analysis.secondary_structure_fraction()
      generated_seq_features["Helix Fraction"].append(helix)
      generated_seq_features["Sheet Fraction"].append(sheet)
      generated_seq_features["Coil Fraction"].append(coil)
    except:
      pass

In [ ]:
# Plot histograms of properties
colors = ['red', 'blue', 'green', 'orange', 'purple']
for key, values in training_seq_features.items():
    plt.figure()
    plt.hist(values)
    plt.title(key)
    plt.xlabel(key)
    plt.ylabel("frequency")
    for i, val in enumerate(generated_seq_features[key]):
      plt.axvline(val, color=colors[i % len(generated_seq_features)], linestyle='dashed', linewidth=2, label=f'{i+1}')

    plt.legend()
    plt.show()

### **AlphaFold**
To fold the proteins, structure prediction using AlphaFold3 on the AlphaFoldServer UI was used: https://alphafoldserver.com/.

### **DSSP**

The PDB outputs of AlphaFold were then used for DSSP, which is used to characterize the secondary structure.

In [ ]:
# ----------------------------------------------
# Secondary Structure Characterization with DSSP
# ----------------------------------------------
# Get all the PDB files
pdb_files = glob.glob("*.pdb")

for file in pdb_files:
  parser = PDBParser()
  structure = parser.get_structure("protein", file)
  model = structure[0]
  dssp = DSSP(model, file)
  helix_codes = {"H", "G", "I", "P"}
  sheet_codes = {"E", "B"}
  helix = sheet = coil = 0

  for aa in dssp:
      ss = aa[2]  # secondary structure code
      if ss in helix_codes:
          helix += 1
      elif ss in sheet_codes:
          sheet += 1
      else:
          coil += 1

  total = helix + sheet + coil
  print(f"DSSP Metrics for {file} \n ===============")
  print(f"Helix: {helix/total:.3f}")
  print(f"Sheet: {sheet/total:.3f}")
  print(f"Coil:  {coil/total:.3f} \n")